# 03 - Erro com pontos flutuantes
Vamos aprender sobre como usar os erros de ponto flutuante para resolução de problemas numéricos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana3
```

Faça o checkout nessa nova branch:

```bash
git checkout semana3
```

<hr />

## Atividade 1
A função exponencial natural pode ser definida pelo limite:
$$
e^x=\lim_{n\to\infty}\left(1+\frac{x}{n}\right)^n,
$$
mas também é dada pela **série de Maclaurin**:
$$
e^x=\sum_{n=0}^{\infty}\frac{x^n}{n!}
=1+\frac{x}{1!}+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
$$

Implemente em **Python** o cálculo de $e^x$ pela série, interrompendo a soma quando o termo ficar menor que o limite prático de contribuição, usando a precisão de máquina como critério.

In [ ]:
import math
import sys

def exp_series(x, atol=0.0):
    """ Aproxima e^x pela série de Maclaurin com critério de parada numérico. """
    eps = sys.float_info.epsilon
    s = 1.0
    term = 1.0
    n = 0
    tol_abs = max(atol, eps)

    while True:
        n += 1
        term *= x / n
        s += term
        if abs(term) < eps * abs(s) or abs(term) < tol_abs:
            break
        if n > 10_000:
            break
    return s, n, term

# Demonstração
for val in [1.0, 5.0, -2.0]:
    approx, nterms, last = exp_series(val)
    print(f"x={val:+g} -> e^x ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, termos={nterms})")

## Atividade 2

Implemente:
$$
e^x\approx\left(1+\frac{x}{n}\right)^n
$$
com $n$ crescente, e:
1. Explique por que, para $x<0$ e $n$ muito grande, pode ocorrer **cancelamento catastrófico**;
2. Proponha um critério de parada numérico para encerrar o crescimento de $n$ sem perder precisão.

In [ ]:
import math
import sys


def exp_limit(x: float, max_steps: int = 256) -> tuple[float, int]:
    if not math.isfinite(x):
        raise ValueError("x deve ser finito.")

    epsilon = sys.float_info.epsilon
    n = max(1, math.floor(-x) + 1) if x < 0 else 1
    previous = None

    for _ in range(max_steps):
        approximation = math.exp(n * math.log1p(x / n))

        if previous is not None:
            tolerance = epsilon * max(abs(approximation), abs(previous))
            if abs(approximation - previous) <= tolerance:
                return approximation, n

        previous = approximation
        n *= 2

    raise RuntimeError("A aproximação não convergiu dentro do limite de passos.")


for value in [1.0, 5.0, -2.0, -20.0]:
    approximation, n = exp_limit(value)
    assert math.isclose(approximation, math.exp(value), rel_tol=10 * sys.float_info.epsilon)
    print(f"x={value:+g}, n={n}, aproximação={approximation:.16g}, math.exp={math.exp(value):.16g}")

## Atividade 3

Para $|x|$ grande, use:
$$
e^x = \left(e^{m\cdot 2^{-k}}\right)^{2^k}, \quad
k = \left\lceil \log_2\!\left(\frac{|x|}{\theta}\right)\right\rceil, \quad m = \frac{x}{2^k}
$$
Calcule $e^{m}$ pela série (Ex. 1) e depois eleve ao quadrado $k$ vezes.

In [ ]:
import math


def exp_scaled(x: float, theta: float = 1.0) -> tuple[float, int]:
    if not math.isfinite(x):
        raise ValueError("x deve ser finito.")
    if not math.isfinite(theta) or theta <= 0:
        raise ValueError("theta deve ser finito e positivo.")

    if abs(x) <= theta:
        k = 0
    else:
        k = math.ceil(math.log2(abs(x)) - math.log2(theta))

    m = math.ldexp(x, -k)
    result, _, _ = exp_series(m)

    for _ in range(k):
        result *= result

    return result, k


for value in [-20.0, -2.0, 0.0, 1.0, 5.0, 20.0]:
    approximation, k = exp_scaled(value)
    assert math.isclose(approximation, math.exp(value), rel_tol=1e-12)
    print(f"x={value:+g}, k={k}, aproximação={approximation:.16g}, math.exp={math.exp(value):.16g}")

## Atividade 4

Use:
$$
\cos x=\sum_{n=0}^{\infty}(-1)^n\frac{x^{2n}}{(2n)!}
$$
com a recursão:
$$
t_{n+1}=t_n\cdot\frac{-x^2}{(2n+1)(2n+2)}
$$
Defina um critério de parada baseado em `epsilon` e compare o erro relativo para $x\in[-20,20]$ (200 pontos) contra `math.cos(x)`.

In [ ]:
import math
import sys


def cos_series(x: float) -> tuple[float, int]:
    epsilon = sys.float_info.epsilon
    terms = [1.0]
    term = 1.0
    n = 0

    while True:
        next_term = term * (-x * x) / ((2 * n + 1) * (2 * n + 2))
        if abs(next_term) <= epsilon * max(1.0, abs(math.fsum(terms))):
            break

        terms.append(next_term)
        term = next_term
        n += 1

    return math.fsum(terms), n


x_values = [-20.0 + 40.0 * index / 199 for index in range(200)]
results = [(x, cos_series(x)[0], math.cos(x)) for x in x_values]
absolute_errors = [(x, abs(approximation - reference))
                   for x, approximation, reference in results]
relative_errors = [
    (x, abs(approximation - reference) / abs(reference))
    for x, approximation, reference in results
    if abs(reference) > sys.float_info.epsilon
]

max_absolute = max(absolute_errors, key=lambda item: item[1])
max_relative = max(relative_errors, key=lambda item: item[1])
print(f"Pontos avaliados: {len(results)}")
print(f"Maior erro absoluto: {max_absolute[1]:.3e} em x={max_absolute[0]:.6g}")
print(f"Maior erro relativo: {max_relative[1]:.3e} em x={max_relative[0]:.6g}")
print(f"Pontos ignorados no erro relativo (|cos(x)| <= epsilon): {len(results) - len(relative_errors)}")

## Atividade 5

Dado $x$ e uma tolerância $\tau$, encontre o menor $N$ tal que:
$$
R_{N+1}(x)=\sum_{n=N+1}^{\infty}\frac{|x|^n}{n!} < \tau
$$

In [1]:
import math


def minimum_n(x: float, tau: float, max_n: int = 100_000) -> tuple[int, float]:
    if not math.isfinite(x):
        raise ValueError("x deve ser finito.")
    if not math.isfinite(tau) or tau <= 0:
        raise ValueError("tau deve ser finito e positivo.")

    absolute_x = abs(x)
    term = 1.0

    for n in range(max_n):
        first_omitted = term * absolute_x / (n + 1)
        ratio = absolute_x / (n + 2)

        if not math.isfinite(first_omitted):
            raise OverflowError("Os termos da série excederam o limite de float.")

        if ratio < 1.0:
            remainder_bound = first_omitted / (1.0 - ratio)
            if remainder_bound < tau:
                return n, remainder_bound

        term = first_omitted

    raise RuntimeError("Não foi encontrado N dentro do limite definido.")


for value in [0.0, 1.0, 5.0, 20.0]:
    tolerance = 1e-10
    n, bound = minimum_n(value, tolerance)
    assert bound < tolerance
    print(f"x={value:g}, tau={tolerance:.1e}, N={n}, limite da cauda={bound:.3e}")

x=0, tau=1.0e-10, N=0, limite da cauda=0.000e+00
x=1, tau=1.0e-10, N=13, limite da cauda=1.229e-11
x=5, tau=1.0e-10, N=28, limite da cauda=2.528e-11
x=20, tau=1.0e-10, N=72, limite da cauda=2.895e-11


## Atividade 6

Usando `decimal` ou `mpmath`, compute $e^x$ em alta precisão e compare com o resultado de `float64` (Ex. 1) para $x\in\{20, 40, 50\}$.
Analise:
- perda de dígitos significativos;
- quando o `float64` começa a saturar por overflow.


In [ ]:
import math
import sys
from decimal import Decimal, localcontext


def exp_series_float64(x: float) -> tuple[float, int]:
    epsilon = sys.float_info.epsilon
    total = 1.0
    term = 1.0
    n = 0

    while True:
        n += 1
        term *= x / n
        total += term
        if abs(term) <= epsilon * abs(total):
            return total, n


with localcontext() as context:
    context.prec = 80
    for x in [20, 40, 50]:
        reference = Decimal(x).exp()
        approximation, terms = exp_series_float64(float(x))
        relative_error = abs(Decimal.from_float(approximation) - reference) / reference
        correct_digits = -math.log10(float(relative_error)) if relative_error else math.inf
        digits_lost = max(0.0, 16.0 - correct_digits)

        print(f"x={x}, float64={approximation:.16g}, Decimal={reference}")
        print(f"  erro relativo={float(relative_error):.3e}, dígitos corretos≈{correct_digits:.2f}, perda≈{digits_lost:.2f}, termos={terms}")

overflow_limit = math.log(sys.float_info.max)
print(f"\nLimite teórico: log(max_float64)={overflow_limit:.15f}")
for x in [overflow_limit - 1e-10, overflow_limit + 1e-10]:
    try:
        print(f"math.exp({x:.15f})={math.exp(x):.6e}")
    except OverflowError:
        print(f"math.exp({x:.15f}) causa OverflowError")

## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 3"
git push origin semana3
```